# 1. Research objective and provisional methodology status

**EXPERIMENTAL UNSUPERVISED ANOMALY MODELLING**

This notebook investigates whether point-in-time Solana DEX wallet snapshots can be ranked by statistical unusualness relative to an earlier reference cohort. It does not change the approved capstone title and does not claim confirmed wash trading, fraud, intent, coordination, or misconduct.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file())
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from solana_risk.models.evaluate_anomaly import evaluate_experiment
from solana_risk.models.train_anomaly import (
    EXPERIMENT_STATUS, MODEL_NAMES, AnomalyConfig, audit_feature_matrix,
    chronological_partitions, fit_experiment, load_snapshots, sha256_file,
)

DATA_PATH = PROJECT_ROOT / 'data' / 'features' / 'wallet_snapshots.csv'
print(EXPERIMENT_STATUS)

## 2. Why supervised training is blocked

The available file contains DEX swap events and no independently verified wash-trading outcome. The historical 249, 18, and nine candidate counts came from behavioural heuristics that overlap with candidate predictors. Treating them as targets would be circular. Supervised performance and calibrated outcome claims therefore remain unavailable.

In [ ]:
snapshots = load_snapshots(DATA_PATH)
audit = audit_feature_matrix(snapshots)
print('Snapshot SHA-256:', sha256_file(DATA_PATH))
print('Labels used:', False)

## 3. Wallet snapshot dataset overview

Each row is one `(wallet, as_of)` observation built only from events strictly before its cutoff. Repeated wallets across dates are cumulative longitudinal observations, not independent samples.

In [ ]:
overview = pd.DataFrame({
    'value': [len(snapshots), audit['feature_count'], snapshots['wallet'].nunique(), audit['single_event_snapshot_count'], audit['insufficient_history_30d_count']]
}, index=['snapshots', 'registered_features', 'unique_wallets', 'single_event_snapshots', 'under_30d_history'])
display(overview)
display(pd.Series(audit['cutoff_counts'], name='snapshot_count').to_frame())

## 4. Model-feature selection

The modelling matrix uses the 29 registry fields in their recorded order. `wallet` and `as_of` remain metadata. Raw identifiers, supplied historical fields, heuristic flags, labels and outcome-like fields are excluded or rejected by the reusable validator.

In [ ]:
from solana_risk.features.build import MODEL_FEATURE_COLUMNS, feature_registry_sha256

print('Registry SHA-256:', feature_registry_sha256())
display(pd.DataFrame({'position': range(1, 30), 'feature': MODEL_FEATURE_COLUMNS}).head(29))
print('Metadata excluded:', {'wallet', 'as_of'}.isdisjoint(MODEL_FEATURE_COLUMNS))

## 5. Missingness and feature distributions

Missing inter-event statistics represent unavailable history for single-event snapshots. They are not filled globally. Correlated fields are retained under the existing registry but recorded as a model limitation.

In [ ]:
display(pd.Series(audit['missing_counts'], name='missing_count').to_frame())
print('Infinite values:', audit['nonfinite_infinite_count'])
print('Training-constant fields:', audit['train_constant_features'])
print('High-correlation pairs:', len(audit['train_high_correlation_pairs']))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
snapshots['historical_event_count'].clip(upper=20).hist(bins=20, ax=axes[0])
axes[0].set(title='Historical event count (clipped at 20)', xlabel='events', ylabel='snapshots')
np.log1p(snapshots['total_usd_volume']).hist(bins=30, ax=axes[1])
axes[1].set(title='Log-transformed total USD volume', xlabel='log1p(USD)')
plt.tight_layout()

## 6. Training and evaluation design

September is the training reference, October is exploratory validation, and December is the final holdout. Full later periods repeat training wallets; later entrants are identity-disjoint diagnostics but differ in observable history. No random row split is used.

In [ ]:
partitions, split = chronological_partitions(snapshots)
display(pd.Series(split.__dict__, name='value').to_frame())
assert split.train_cutoff < split.validation_cutoff < split.holdout_cutoff
assert set(partitions['train']['wallet']).isdisjoint(set(partitions['validation_new_wallets']['wallet']))

## 7. Train-only preprocessing

Median imputation is inside every fitted pipeline. LOF also uses a train-fitted robust scaler. Isolation Forest remains unscaled, while the statistical baseline learns train medians and MADs.

In [ ]:
config = AnomalyConfig()
bundle = fit_experiment(snapshots, config=config, snapshot_sha256=sha256_file(DATA_PATH))
evaluation, scores = evaluate_experiment(bundle, snapshots)
display(pd.Series(bundle['metadata']['config'], name='value').to_frame())
for name, pipeline in bundle['models'].items():
    print(name, 'imputer fitted feature count:', len(pipeline.named_steps['imputer'].statistics_))

## 8. Isolation Forest experiment

Isolation Forest is seeded and configured explicitly. Its `contamination='auto'` setting is an algorithm convention, not an estimate of misconduct prevalence. The reusable scorer negates sklearn's normality score so larger always means more unusual.

In [ ]:
iforest_summary = pd.DataFrame({
    period: evidence['isolation_forest']
    for period, evidence in evaluation['score_distributions'].items()
}).T[['count', 'median', 'q95', 'q99', 'max']]
display(iforest_summary)

## 9. Local Outlier Factor experiment

LOF uses robust scaling and `novelty=True` to score later snapshots. The fitted training attribute `negative_outlier_factor_` and novelty-mode `score_samples` are different quantities; the training intrinsic distribution is reported separately.

In [ ]:
display(pd.Series(evaluation['lof_score_contract']['training_intrinsic_summary'], name='training_intrinsic').to_frame())
display(pd.Series(evaluation['score_distributions']['holdout']['local_outlier_factor'], name='holdout_novelty').to_frame())
print(evaluation['lof_score_contract']['novelty_scoring'])

## 10. Robust statistical baseline

The baseline computes clipped feature-level deviations from train medians using train MAD. A changed train-constant dimension receives a fixed penalty; unchanged values contribute zero. Root-mean-square aggregation is interpretable but can count correlated feature families repeatedly.

In [ ]:
display(pd.DataFrame({
    period: evidence['robust_mad']
    for period, evidence in evaluation['score_distributions'].items()
}).T[['count', 'median', 'q95', 'q99', 'max']])

## 11. Anomaly-score visualizations

Score magnitudes are method-specific. These panels therefore show distributions separately rather than placing unlike scales on one axis.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, name in zip(axes, MODEL_NAMES):
    values = scores['holdout'][name]
    if name == 'local_outlier_factor':
        values = np.log1p(values)
        xlabel = 'log1p(score)'
    else:
        xlabel = 'score'
    ax.hist(values, bins=35)
    ax.set(title=name.replace('_', ' ').title(), xlabel=xlabel, ylabel='wallet snapshots')
plt.tight_layout()

## 12. Model comparison

Rank agreement describes whether detectors prioritize similar observations; it is not evidence that either list is correct.

In [ ]:
agreement = pd.Series(evaluation['rank_agreement']['holdout'], name='holdout_spearman').to_frame()
display(agreement)
display(pd.DataFrame(evaluation['review_budget_sensitivity']['holdout']).T[['effective_budget', 'all_model_intersection_count']])

## 13. Hyperparameter sensitivity

Alternatives are refitted on September and compared on October only. December is not used for parameter choice.

In [ ]:
sensitivity = pd.DataFrame(evaluation['validation_parameter_sensitivity']).T
display(sensitivity[['model', 'changed_parameter', 'validation_rank_correlation_with_primary', 'evaluated_on_holdout']])

## 14. Effects of sparse wallet histories

Single-event and under-30-day histories are observational-coverage conditions. Score differences across these groups may reflect what is missing rather than risky behaviour.

In [ ]:
rows = []
for cohort in ('single_event_history', 'insufficient_history_30d'):
    evidence = evaluation['holdout_cohort_diagnostics'][cohort]
    for model in MODEL_NAMES:
        rows.append({
            'cohort': cohort, 'model': model,
            'selected_count': evidence['selected_count'],
            'selected_median': evidence['selected_scores'][model]['median'],
            'other_median': evidence['other_scores'][model]['median'],
        })
display(pd.DataFrame(rows))

## 15. Temporal stability and drift

Repeated-wallet rank correlation measures longitudinal consistency, not independent generalisation. Entrant drift is assessed separately because new wallets have shorter observable histories.

In [ ]:
temporal = evaluation['temporal_score_changes']['train_to_holdout']
display(pd.DataFrame({name: temporal[name] for name in MODEL_NAMES}).T[['rank_correlation']])
entrant_shift = pd.DataFrame(evaluation['feature_shift']['holdout_new_wallets_vs_train']).T
display(entrant_shift.sort_values('absolute_median_shift_in_train_iqr', ascending=False, na_position='first').head(10))

## 16. Interpretation of unverified anomaly candidates

A top-k list is an operationally bounded set of **unverified candidates for investigation**. It must not be called a detected-fraud list. Detector agreement is not label correctness, and score thresholds do not reveal prevalence.

In [ ]:
budget_evidence = evaluation['review_budget_sensitivity']['holdout']
candidate_summary = pd.DataFrame({
    budget: {
        'effective_budget': item['effective_budget'],
        'all_model_intersection_count': item['all_model_intersection_count'],
    } for budget, item in budget_evidence.items()
}).T.sort_index(key=lambda x: x.astype(int))
display(candidate_summary)
print('No wallet identifiers are displayed; no case has verified outcome evidence.')

## 17. Research limitations

There are no verified outcomes; later full cutoffs reuse wallets and cumulative history; entrant cohorts are sparse and shifted; highly correlated fields can overweight behavioural families; extreme legitimate volume can look unusual; direct counterparty edges and NFT events are absent; and the dataset provider, extraction query, licence, and sampling process remain unresolved.

In [ ]:
for limitation in evaluation['limitations']:
    print('-', limitation)

## 18. Implications for the approved capstone

This phase implements reproducible unsupervised DEX anomaly ranking. Anomaly scores do not establish that a wallet conducted wash trading. API, frontend, container, and cloud work remain outside this phase and require separate instruction.

In [ ]:
print(EXPERIMENT_STATUS)
print('Conclusion: reproducible anomaly-ranking evidence only; no verified wash-trading conclusion.')